# EXP-001 — honest prefix-to-tail validation

## tl;dr

This experiment creates the validation anchor for ROGII. Every training well is converted into four pseudo-test cases by exposing only an early TVT prefix and evaluating predictions on the hidden tail. Public-test wells, leaderboard scores, and train/test overlap lookup are not used for selection.

The notebook compares deliberately simple, reproducible baselines before the expensive production pipeline is adapted to this protocol.

## Context & Methods

### Key assumptions

- Rows are ordered along measured depth (`MD`).
- At inference time a contiguous TVT prefix is known and the remaining tail is hidden.
- A useful improvement must work across several prefix lengths, not only one cut.
- Primary metric is pooled row-level RMSE; median and p90 well-case RMSE are guardrails.

Cuts: 50%, 65%, 75%, and 85% visible TVT. Candidates use only the visible prefix:

- `last_value`: constant continuation.
- `local_linear`: least-squares TVT trend over the last 300 visible rows.
- `robust_slope`: median recent dTVT/dMD continuation.
- `formation_shape`: physically motivated `formation - Z` shape, with formation selected by an inner prefix holdout.
- `prefix_selector`: selects the best candidate using an inner 80/20 split inside the visible prefix.

In [1]:
from pathlib import Path
import json
import math
import time

import numpy as np
import pandas as pd

SEED = 42
CUT_FRACTIONS = (0.50, 0.65, 0.75, 0.85)
FORMATIONS = ("ANCC", "ASTNU", "ASTNL", "EGFDU", "EGFDL", "BUDA")
LOCAL_WINDOW = 300
MIN_VISIBLE = 100
MIN_HIDDEN = 50

np.random.seed(SEED)

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        q = p / 'competitions/public-comp/wellbore-geology-prediction'
        if (q / 'datasets/train').is_dir():
            return q
        if (p / 'datasets/train').is_dir() and p.name == 'wellbore-geology-prediction':
            return p
    raise FileNotFoundError('Run from the MLKaggleTasks repository or the competition directory')

PROJECT_ROOT = find_project_root()
TRAIN_DIR = PROJECT_ROOT / 'datasets/train'
RESULTS_DIR = PROJECT_ROOT / 'experiments/exp_001_prefix_validation/results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
WELL_FILES = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
print('project:', PROJECT_ROOT)
print('train wells:', len(WELL_FILES))
assert len(WELL_FILES) >= 700, 'Unexpectedly incomplete training data'


project: /Users/flexonafft/MLKaggleTasks/competitions/public-comp/wellbore-geology-prediction
train wells: 773


## Data

Validate the schema and build deterministic pseudo-test cases. The true `TVT` tail is retained only for scoring.

In [2]:
required = {'MD', 'Z', 'TVT', *FORMATIONS}
schema_rows = []
for path in WELL_FILES:
    df = pd.read_csv(path, nrows=5)
    missing = sorted(required - set(df.columns))
    schema_rows.append({'well': path.name[:8], 'missing': ','.join(missing)})
schema = pd.DataFrame(schema_rows)
assert (schema['missing'] == '').all(), schema[schema['missing'] != ''].head()
schema.head()

,well,missing
0,000d7d20,
1,00bbac68,
2,00e12e8b,
3,015fe0d2,
4,01869cd4,


## Baseline candidates

All fitting functions receive only rows before the outer cut. `formation_shape` chooses its geological reference using a smaller validation tail inside that visible prefix.

In [3]:
def rmse(y, pred):
    y = np.asarray(y, dtype=float)
    pred = np.asarray(pred, dtype=float)
    return float(np.sqrt(np.mean((y - pred) ** 2)))


def predict_candidates(df, n_visible):
    visible = df.iloc[:n_visible]
    future = df.iloc[n_visible:]
    x_vis = visible['MD'].to_numpy(float)
    y_vis = visible['TVT'].to_numpy(float)
    x_future = future['MD'].to_numpy(float)
    predictions = {}

    predictions['last_value'] = np.full(len(future), y_vis[-1])

    n_local = min(LOCAL_WINDOW, len(visible))
    x_local = x_vis[-n_local:]
    y_local = y_vis[-n_local:]
    x0 = x_local[-1]
    slope, intercept = np.polyfit(x_local - x0, y_local, 1)
    predictions['local_linear'] = intercept + slope * (x_future - x0)

    dx = np.diff(x_local)
    dy = np.diff(y_local)
    valid = np.isfinite(dx) & np.isfinite(dy) & (np.abs(dx) > 1e-9)
    robust_slope = float(np.median(dy[valid] / dx[valid])) if valid.any() else 0.0
    predictions['robust_slope'] = y_vis[-1] + robust_slope * (x_future - x_vis[-1])

    inner_cut = max(MIN_VISIBLE, int(round(n_visible * 0.80)))
    inner_cut = min(inner_cut, n_visible - MIN_HIDDEN)
    best_formation = FORMATIONS[0]
    best_score = math.inf
    for formation in FORMATIONS:
        shape = (df[formation] - df['Z']).to_numpy(float)
        offset = float(np.median(y_vis[:inner_cut] - shape[:inner_cut]))
        score = rmse(y_vis[inner_cut:], shape[inner_cut:n_visible] + offset)
        if score < best_score:
            best_score = score
            best_formation = formation
    shape = (df[best_formation] - df['Z']).to_numpy(float)
    offset = float(np.median(y_vis - shape[:n_visible]))
    predictions['formation_shape'] = shape[n_visible:] + offset
    return predictions, best_formation


def select_by_inner_prefix(df, n_visible):
    inner_visible = max(MIN_VISIBLE, int(round(n_visible * 0.80)))
    inner_visible = min(inner_visible, n_visible - MIN_HIDDEN)
    inner_predictions, _ = predict_candidates(df.iloc[:n_visible].reset_index(drop=True), inner_visible)
    y_inner = df['TVT'].iloc[inner_visible:n_visible].to_numpy(float)
    scores = {name: rmse(y_inner, pred) for name, pred in inner_predictions.items()}
    return min(scores, key=scores.get), scores


## Run validation

One record is produced for every `(well, cut, model)` combination. Failures are recorded rather than silently dropped.

In [4]:
started = time.time()
records = []
failures = []

for well_number, path in enumerate(WELL_FILES, 1):
    well = path.name.split('__')[0]
    try:
        df = pd.read_csv(path).sort_values('MD').reset_index(drop=True)
        df = df.dropna(subset=['MD', 'Z', 'TVT']).reset_index(drop=True)
        for cut_fraction in CUT_FRACTIONS:
            n_visible = int(round(len(df) * cut_fraction))
            if n_visible < MIN_VISIBLE or len(df) - n_visible < MIN_HIDDEN:
                failures.append({'well': well, 'cut_fraction': cut_fraction, 'reason': 'too_short'})
                continue
            predictions, formation = predict_candidates(df, n_visible)
            selected_name, inner_scores = select_by_inner_prefix(df, n_visible)
            predictions['prefix_selector'] = predictions[selected_name]
            y_true = df['TVT'].iloc[n_visible:].to_numpy(float)
            for model, pred in predictions.items():
                records.append({
                    'well': well,
                    'cut_fraction': cut_fraction,
                    'model': model,
                    'n_visible': n_visible,
                    'n_hidden': len(y_true),
                    'rmse': rmse(y_true, pred),
                    'bias': float(np.mean(pred - y_true)),
                    'selected_model': selected_name if model == 'prefix_selector' else '',
                    'selected_formation': formation if model == 'formation_shape' else '',
                    'squared_error_sum': float(np.sum((pred - y_true) ** 2)),
                })
    except Exception as exc:
        failures.append({'well': well, 'cut_fraction': None, 'reason': repr(exc)})
    if well_number % 100 == 0:
        print(f'{well_number}/{len(WELL_FILES)} wells')

results = pd.DataFrame(records)
failures_df = pd.DataFrame(failures)
elapsed = time.time() - started
print(f'completed {results.well.nunique()} wells, {len(results)} scored cases in {elapsed:.1f}s; failures={len(failures_df)}')
assert results.well.nunique() >= 700
results.head()

100/773 wells


200/773 wells


300/773 wells


400/773 wells


500/773 wells


600/773 wells


700/773 wells


completed 773 wells, 15460 scored cases in 5.8s; failures=0


,well,cut_fraction,model,n_visible,n_hidden,rmse,bias,selected_model,selected_formation,squared_error_sum
0,000d7d20,0.5,last_value,2639,2639,4.533036,2.619693,,,5.422727e+04
1,000d7d20,0.5,local_linear,2639,2639,31.884165,-28.474942,,,2.682807e+06
2,000d7d20,0.5,robust_slope,2639,2639,26.829309,-23.780307,,,1.899583e+06
3,000d7d20,0.5,formation_shape,2639,2639,0.004971,0.000038,,ASTNU,6.520000e-02
4,000d7d20,0.5,prefix_selector,2639,2639,0.004971,0.000038,formation_shape,,6.520000e-02


## Results

Pooled RMSE weights every hidden row equally, matching the competition metric. Well-case quantiles expose instability that pooled RMSE alone can hide.

In [5]:
def summarize(group):
    return pd.Series({
        'pooled_rmse': np.sqrt(group['squared_error_sum'].sum() / group['n_hidden'].sum()),
        'mean_well_rmse': group['rmse'].mean(),
        'median_well_rmse': group['rmse'].median(),
        'p90_well_rmse': group['rmse'].quantile(0.90),
        'worst_well_rmse': group['rmse'].max(),
        'well_cases': len(group),
        'hidden_rows': group['n_hidden'].sum(),
    })

summary = results.groupby('model', sort=False).apply(summarize, include_groups=False).reset_index()
summary = summary.sort_values('pooled_rmse').reset_index(drop=True)
by_cut = results.groupby(['cut_fraction', 'model'], sort=False).apply(summarize, include_groups=False).reset_index()
winner_counts = (results.loc[results.groupby(['well', 'cut_fraction'])['rmse'].idxmin()]
                 .groupby('model').size().rename('wins').reset_index().sort_values('wins', ascending=False))
display(summary.round(4))
display(by_cut.pivot(index='model', columns='cut_fraction', values='pooled_rmse').round(4))
display(winner_counts)

,model,pooled_rmse,mean_well_rmse,median_well_rmse,p90_well_rmse,worst_well_rmse,well_cases,hidden_rows
0,formation_shape,0.0054,0.0054,0.0053,0.0057,0.0095,3092.0,6365301.0
1,prefix_selector,0.0054,0.0054,0.0053,0.0057,0.0095,3092.0,6365301.0
2,last_value,11.4937,8.6360,7.0537,16.8669,58.1652,3092.0,6365301.0
3,robust_slope,31.2987,18.4268,12.1532,41.0586,240.9127,3092.0,6365301.0
4,local_linear,31.6612,18.7576,12.4237,42.1399,263.1030,3092.0,6365301.0


cut_fraction,0.50,0.65,0.75,0.85
model,,,,
formation_shape,0.0054,0.0054,0.0054,0.0053
last_value,12.6456,11.5766,10.4752,8.4999
local_linear,40.9969,28.1763,20.5643,13.9237
prefix_selector,0.0054,0.0054,0.0054,0.0053
robust_slope,40.8320,27.2802,20.2491,13.6404


,model,wins
0,formation_shape,3092


In [6]:
selector_choices = (results[results['model'] == 'prefix_selector']['selected_model']
                    .value_counts(dropna=False).rename_axis('selected_model').reset_index(name='cases'))
formation_choices = (results[results['model'] == 'formation_shape']['selected_formation']
                     .value_counts(dropna=False).rename_axis('formation').reset_index(name='cases'))
display(selector_choices)
display(formation_choices)

,selected_model,cases
0,formation_shape,3092


,formation,cases
0,ANCC,697
1,BUDA,603
2,ASTNU,532
3,ASTNL,503
4,EGFDL,398
5,EGFDU,359


## Takeaways

This harness is valid as a computation pipeline, but the formation result is intentionally treated as optimistic: the supplied same-well formation surfaces encode the train TVT shape almost exactly. The next experiment must simulate train/test version mismatch by perturbing or transferring formation/contact geometry. Only after that stress test should production ML/PF candidates be promoted.

In [7]:
results.to_csv(RESULTS_DIR / 'per_well_cut_metrics.csv', index=False)
summary.to_csv(RESULTS_DIR / 'summary.csv', index=False)
by_cut.to_csv(RESULTS_DIR / 'summary_by_cut.csv', index=False)
winner_counts.to_csv(RESULTS_DIR / 'winner_counts.csv', index=False)
selector_choices.to_csv(RESULTS_DIR / 'selector_choices.csv', index=False)
formation_choices.to_csv(RESULTS_DIR / 'formation_choices.csv', index=False)
failures_df.to_csv(RESULTS_DIR / 'failures.csv', index=False)
run_info = {
    'experiment_id': 'exp_001',
    'seed': SEED,
    'cut_fractions': list(CUT_FRACTIONS),
    'wells': int(results.well.nunique()),
    'scored_rows': int(len(results)),
    'elapsed_sec': elapsed,
    'best_model': str(summary.iloc[0]['model']),
    'best_pooled_rmse': float(summary.iloc[0]['pooled_rmse']),
}
(RESULTS_DIR / 'run.json').write_text(json.dumps(run_info, indent=2) + '\n')
print(json.dumps(run_info, indent=2))
print('saved:', sorted(p.name for p in RESULTS_DIR.iterdir()))

{
  "experiment_id": "exp_001",
  "seed": 42,
  "cut_fractions": [
    0.5,
    0.65,
    0.75,
    0.85
  ],
  "wells": 773,
  "scored_rows": 15460,
  "elapsed_sec": 5.786854028701782,
  "best_model": "formation_shape",
  "best_pooled_rmse": 0.005370068041757369
}
saved: ['failures.csv', 'formation_choices.csv', 'per_well_cut_metrics.csv', 'run.json', 'selector_choices.csv', 'summary.csv', 'summary_by_cut.csv', 'winner_counts.csv']
